In [12]:
import json
import os
import zipfile
import tqdm
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [2]:
def load_results_from_file(file_path : str):
    with open(file_path, 'r') as f:
        return json.load(f)
    
def load_results_from_folder(folder_path : str):
    results = []
    for file_name in tqdm.tqdm(os.listdir(folder_path), desc="Loading results from folder"):
        if file_name.endswith('.json'):
            with open(os.path.join(folder_path, file_name), 'r') as f:
                results.append(json.load(f))
    return results

def load_results_from_zip(zip_path : str):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        results = []
        for file_name in tqdm.tqdm(file_list, desc="Loading results from zip"):
            if file_name.endswith('.json'):
                with zip_ref.open(file_name) as f:
                    results.append(json.load(f))
        return results

In [3]:
def verifyJson(jsonData: dict) -> list:
    results = jsonData.get("results", None)
    if results is None:
        return ["Missing 'results' key in JSON data."]
    solverResult = results.get("solver",{}).get("score", 0)
    if solverResult < 0:
        return [f"Solver score is negative: {solverResult}"]
    
    """ mctsResultsList = results.get("mcts",{}).get("tries", [])
    if not mctsResultsList:
        return ["Missing 'tries' key in 'mcts' results or it is empty."]    
    
    for isx, mctsTry in enumerate(mctsResultsList):
        for step, stepData in enumerate(mctsTry.get("steps", [])):
            stepScore = stepData.get("score", None)
            if stepScore is None:
                return [f"Missing 'score' key in step {step} of try {isx}."]
            if stepScore < 0:
                return [f"Step score is negative in step {step} of try {isx}: {stepScore}"]
             """
    return []

In [4]:
def check_results_from_zip(zip_path):
    error = {}
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        json_files = [f for f in zip_ref.namelist() if f.endswith('.json')]
        
        for json_file in tqdm.tqdm(json_files, desc="Checking JSON files"):
            with zip_ref.open(json_file) as f:
                data = json.load(f)
                errors = verifyJson(data)
                if errors:
                    error[json_file] = errors
    if error:
        for file, errs in error.items():
            print(f"Invalid JSON in file: {file.split('/')[-1].split('_')[1]}")
            for err in errs:
                print(f"  - {err}")
        return False
    print("All JSON files are valid.")
    return True

In [5]:
zip_path = '../results/experiments_13-05-2026_13-07-52.zip'  # Path to your zip file
# lst = check_results_from_zip(zip_path)
# print(lst)

### Gather all data + analyse them

In [6]:
zip_path = '../results/experiments_13-05-2026_13-07-52.zip'
loaded_results = load_results_from_zip(zip_path)
print(f"Loaded {len(loaded_results)} results from zip file.")

Loading results from zip: 100%|██████████| 77266/77266 [01:21<00:00, 943.90it/s] 


Loaded 77265 results from zip file.


In [7]:
class DataExtractor : 
    def __init__(self, dataset : list):
        self.agentsInterval = (-1, -1) # (min, max) of numAgents
        self.objectsInterval = (-1, -1) # (min, max) of numObjects
        self.seedsInterval = (-1, -1) # (min, max) of seed
        self.ratioRandomInterval = (-1.0, -1.0) # (min, max) of ratioRandom
        self.scores = {} # {parameters -> {"solver": score, "mcts": [scores]}}
        self.times = {} # {parameters -> {"solver": time, "mcts": times}}
        self.allocations = {} #parameters -> {"solver": allocation, "mcts": allocation}
        self.metrics = {} # {parameters -> {"solver": {EF : value, ...}, "mcts": [{EF : value, ...}]}}
        self.parametersOrder = () # List of parameters keys to maintain order
        self.__extract(dataset)
    def __extract(self, dataset : list):
        for data in dataset:
            parameters = data.get("parameters", {})
            #set intervals
            numAgents = parameters.get("numAgents", None)
            numObjects = parameters.get("numObjects", None)
            seed = parameters.get("seed", None)
            ratioRandom = parameters.get("ratioRandom", None)
            if self.agentsInterval == (-1, -1):
                self.agentsInterval = (numAgents, numAgents) 
            else:
                self.agentsInterval = (min(self.agentsInterval[0], numAgents), max(self.agentsInterval[1], numAgents))
            if self.objectsInterval == (-1, -1):
                self.objectsInterval = (numObjects, numObjects)
            else:
                self.objectsInterval = (min(self.objectsInterval[0], numObjects), max(self.objectsInterval[1], numObjects))
            if self.seedsInterval == (-1, -1):
                self.seedsInterval = (seed, seed)
            else:
                self.seedsInterval = (min(self.seedsInterval[0], seed), max(self.seedsInterval[1], seed))
            if self.ratioRandomInterval == (-1.0, -1.0):
                self.ratioRandomInterval = (ratioRandom, ratioRandom)
            else :
                self.ratioRandomInterval = (min(self.ratioRandomInterval[0], ratioRandom), max(self.ratioRandomInterval[1], ratioRandom))
            
            if len(self.parametersOrder) == 0:
                self.parametersOrder = tuple(parameters.keys())
                
            param_key = self.parameters_to_key(parameters)
            
            if param_key not in self.scores:
                self.scores[param_key] = {"solver": None, "mcts": []}
                self.times[param_key] = {"solver": None, "mcts": []}
                self.allocations[param_key] = {"solver": None, "mcts": []}
                self.metrics[param_key] = {"solver": {}, "mcts": []}
            
            # Extract solver results
            results = data.get("results", {})
            solver_result = results.get("solver", {})
            self.scores[param_key]["solver"] = solver_result.get("score", None)
            self.times[param_key]["solver"] = solver_result.get("time", None)
            self.allocations[param_key]["solver"] = solver_result.get("allocation", None)
            self.metrics[param_key]["solver"] = solver_result.get("metrics", {})
            # Extract MCTS results
            mcts_results = results.get("mcts", {}).get("tries", [])
            mcts_global_scores = []
            mcts_global_times = []
            mcts_global_allocations = []
            mcts_global_metrics = []
            for mcts_try in mcts_results:
                steps = mcts_try.get("steps", [])
                local_scores = []
                local_times = []
                local_allocations = []
                local_metrics = []
                for step in steps:
                    local_scores.append(step.get("score", None))
                    local_times.append(step.get("stepTimeUs", None))
                    local_allocations.append(step.get("allocation", None))
                    local_metrics.append(step.get("metrics", {}))
                mcts_global_scores.append(local_scores)
                mcts_global_times.append(local_times)
                mcts_global_allocations.append(local_allocations)
                mcts_global_metrics.append(local_metrics)
            self.scores[param_key]["mcts"] = mcts_global_scores
            self.times[param_key]["mcts"] = mcts_global_times
            self.allocations[param_key]["mcts"] = mcts_global_allocations
            self.metrics[param_key]["mcts"] = mcts_global_metrics
        
    def parameters_to_key(self, parameters : dict) -> str:
        """
        Transform dict to a string to simplify and uniformized gathering

        Args:
            parameters (dict): A dict with parameter keys and values

        Returns:
            str: A string representation of the parameters dict, with key-value pairs joined by underscores.
        """
        return "_".join(f"{v}" for v in parameters.values())
    
    def key_to_parameters(self, key : str) -> dict:
        """
        Transform a string key back to a parameters dict.

        Args:
            key (str): A string representation of the parameters, with values joined by underscores.

        Returns:
            dict: A dict with parameter keys and values extracted from the string key.
        """
        values = key.split("_")
        return {param_key: self.__cast_value(values[i]) for i, param_key in enumerate(self.parametersOrder)}
    def __cast_value(self, value : str):
        """
        Cast a string value to int, float, or keep as string based on its content.

        Args:
            value (str): The string value to cast.
        Returns:
            int, float, or str: The casted value as int, float, or original
        """
        try:
            return int(value)
        except ValueError:
            pass
        try:
            return float(value)
        except ValueError:
            pass
        return value
    
    def get_scores(self):
        return self.scores
    
    def get_times(self):
        return self.times
    
    def get_allocations(self):
        return self.allocations
    
    def get_metrics(self):
        return self.metrics
    
    def get_parameters_order(self):
        """
        <p>
        Get the order of parameters keys.
        </p>
        <p>
        For exemple: ["numAgents", "numObjects", "seed", "ratioRandom", "budget"]
        </p>
        Returns:
            list: A list of parameter keys in the order they were first encountered.
        """
        return self.parametersOrder
    
    def get_intervals(self):
        return {
            "numAgents": self.agentsInterval,
            "numObjects": self.objectsInterval,
            "seed": self.seedsInterval,
            "ratioRandom": self.ratioRandomInterval
        }
    
    def get_filtered_data(self, functionName : str, numAgents : int = -1, numObjects : int = -1, seed : int = -1, ratioRandom : float = -1.0, howMuchData : int = -1, suffle : bool = False, seedSuffle : int =42):
        """
        <p>
        Get filtered data based on provided parameters. If a parameter is None, it is not used for filtering.
        </p>
        <p>
        "-1" for int parameters and "-1.0" for float parameters are used to indicate that the parameter should not be used for filtering.
        Args:
            functionName (str): The function name to filter by. Can only be a getter function from this class
            numAgents (int, optional): Number of agents to filter by. Defaults to -1.
            numObjects (int, optional): Number of objects to filter by. Defaults to -1.
            seed (int, optional): Seed value to filter by. Defaults to -1.
            ratioRandom (float, optional): Ratio of random to filter by. Defaults to -1.0.
            howMuchData (int, optional): The number of data points to return. Defaults to -1.
            suffle (bool, optional): Whether to return data randomly if howMuchData is specified. Defaults to False.
            seedsuffle (int, optional): The seed for the random number generator. Defaults to 42.
        Returns:
            dict: A dictionary containing filtered scores, times, allocations, and metrics based on the provided parameters.
        """
        filtered_data = {}
        f = None;
        # verify that the function is a getter function from this class
        match functionName :
            case "get_scores":
                f = self.get_scores
            case "get_times":
                f = self.get_times
            case "get_allocations":
                f = self.get_allocations
            case "get_metrics":
                f = self.get_metrics
            case _:
                raise ValueError(f"Invalid function name: {functionName}. Must be one of 'get_scores', 'get_times', 'get_allocations', 'get_metrics'.")
        
        # verify that the parameters are valid
        if numAgents != -1 and (numAgents < self.agentsInterval[0] or numAgents > self.agentsInterval[1]):
            raise ValueError(f"numAgents must be between {self.agentsInterval[0]} and {self.agentsInterval[1]}.")
        if numObjects != -1 and (numObjects < self.objectsInterval[0] or numObjects > self.objectsInterval[1]):
            raise ValueError(f"numObjects must be between {self.objectsInterval[0]} and {self.objectsInterval[1]}.")
        if seed != -1 and (seed < self.seedsInterval[0] or seed > self.seedsInterval[1]):
            raise ValueError(f"seed must be between {self.seedsInterval[0]} and {self.seedsInterval[1]}.")
        if ratioRandom != -1.0 and (ratioRandom < self.ratioRandomInterval[0] or ratioRandom > self.ratioRandomInterval[1]):
            raise ValueError(f"ratioRandom must be between {self.ratioRandomInterval[0]} and {self.ratioRandomInterval[1]}.")   
        if howMuchData != -1 and howMuchData < 0:
            raise ValueError(f"howMuchData must be a positive integer or -1.")
        
        items = f().items()
        if howMuchData != -1 and suffle:
            items = list(items)
            randomizer = random.Random(seedSuffle)
            randomizer.shuffle(items)
        
        count = 0
        for param_key, data in items:
            # Check if the parameter values match the filter criteria
            if (numAgents == -1 or self.key_to_parameters(param_key).get("numAgents") == numAgents) and \
               (numObjects == -1 or self.key_to_parameters(param_key).get("numObjects") == numObjects) and \
               (seed == -1 or self.key_to_parameters(param_key).get("seed") == seed) and \
               (ratioRandom == -1.0 or self.key_to_parameters(param_key).get("ratioRandom") == ratioRandom):
                filtered_data[param_key] = data
                count += 1
                if howMuchData != -1 and count >= howMuchData:
                    break
                
        return filtered_data


In [8]:
data_extractor = DataExtractor(loaded_results)

In [9]:
print("Extracted Scores:")
for param_key, score_data in data_extractor.get_filtered_data("get_scores", numAgents=5, howMuchData=5, suffle=True).items():
    print(f"Parameters: {param_key}")
    print(f"  Solver Score: {score_data['solver']}")
    print(f"  MCTS Scores: {score_data['mcts']}")
    

Extracted Scores:
Parameters: 5_28_42_0.77_140
  Solver Score: 18.6043731822
  MCTS Scores: [[17.6172330243, 17.6172330243, 17.6172330243, 17.6172330243, 17.6172330243, 17.6172330243, 17.6172330243, 17.6172330243, 17.9207247728, 17.9207247728], [17.3890601216, 17.6244106938, 17.789611212, 17.789611212, 17.789611212, 17.789611212, 17.789611212, 17.789611212, 17.789611212, 17.789611212], [17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768, 17.7843940768], [17.7117782444, 17.7117782444, 17.7117782444, 17.7117782444, 17.7117782444, 17.7117782444, 17.7211131504, 17.7211131504, 17.7211131504, 17.7211131504], [17.5806291466, 17.5806291466, 17.5806291466, 17.5806291466, 17.5806291466, 17.5806291466, 17.5806291466, 17.5806291466, 17.5923497546, 17.6779030413], [17.2326444412, 17.6855550509, 17.6855550509, 17.6855550509, 17.6855550509, 17.6855550509, 17.6855550509, 17.6855550509, 17.6855550509, 17.7895691013], [17

In [10]:
#normalized_solver_scores to 1 and score to %
normalized_scores = {}
for param_key, score_data in data_extractor.get_scores().items():
    solver_score = score_data["solver"]
    mcts_scores = score_data["mcts"]
    if solver_score is not None and solver_score != 0:
        normalized_solver_score = 1.0
        normalized_mcts_scores = [[mcts_score / solver_score if solver_score != 0 else None for mcts_score in mcts_try] for mcts_try in mcts_scores]
        normalized_scores[param_key] = {"solver": normalized_solver_score, "mcts": normalized_mcts_scores}
    else:
        normalized_scores[param_key] = {"solver": None, "mcts": None}
    
print("Normalized Scores:")
i = 0
for param_key, score_data in normalized_scores.items():
    print(f"  {param_key}: {score_data}")
    i += 1
    if (i == 5):
        break
               


Normalized Scores:
  5_5_44_0.92_25: {'solver': 1.0, 'mcts': [[-0.6175993497005802, -0.6175993497005802, -0.6175993497005802, -0.6175993497005802, -0.6175993497005802, 0.8380919721269522, 0.8380919721269522, 0.8380919721269522, 0.8380919721269522, 0.8380919721269522], [-0.6268488616829186, -0.6268488616829186, -0.6268488616829186, 0.6653089549642858, 0.6653089549642858, 0.6653089549642858, 0.6653089549642858, 0.6653089549642858, 0.6653089549642858, 0.6653089549642858], [-0.667099507705957, -0.667099507705957, 0.7548498534393185, 0.7548498534393185, 0.7548498534393185, 0.7548498534393185, 0.7548498534393185, 0.7548498534393185, 0.7548498534393185, 0.7548498534393185], [-0.5608251495199774, -0.5608251495199774, -0.5608251495199774, -0.5608251495199774, -0.5608251495199774, -0.5608251495199774, 0.84521321953226, 0.84521321953226, 0.9029751525680748, 0.9029751525680748], [-2.2147544996623076, -0.7498136679983212, -0.7498136679983212, -0.6843423519921582, -0.6632040609949561, -0.66320406099

In [18]:
%pip install -q plotly nbformat

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [20]:
scores = data_extractor.get_filtered_data("get_scores", numAgents=5)
normalized_scores = {}
for param_key, score_data in scores.items():
    solver_score = score_data["solver"]
    mcts_scores = score_data["mcts"]
    if solver_score is not None and solver_score != 0:
        normalized_solver_score = 1.0
        normalized_mcts_scores = [[mcts_score / solver_score for mcts_score in mcts_try] for mcts_try in mcts_scores]
        normalized_scores[param_key] = {"solver": normalized_solver_score, "mcts": normalized_mcts_scores}
    else:
        normalized_scores[param_key] = {"solver": None, "mcts": None}

# Graphe interactif: survol souris pour afficher les paramètres
import pandas as pd
import plotly.graph_objects as go

def to_scalar(x):
    if isinstance(x, (list, tuple)):
        return x[-1] if len(x) > 0 else np.nan
    return x

param_keys = sorted(normalized_scores.keys(), key=str)

mcts_points = []
mcts_mean_points = []
solver_points = []

for i, k in enumerate(param_keys):
    entry = normalized_scores[k]
    solver_val = entry["solver"] if entry["solver"] is not None else np.nan

    solver_points.append({
        "idx": i,
        "param_key": k,
        "series": "Solver",
        "value": solver_val
    })

    mcts_tries = entry["mcts"] if entry["mcts"] is not None else []
    try_vals = [to_scalar(t) for t in mcts_tries]
    try_vals = [v for v in try_vals if v is not None and not np.isnan(v)]

    for t_idx, v in enumerate(try_vals):
        mcts_points.append({
            "idx": i,
            "param_key": k,
            "series": "MCTS try",
            "value": v,
            "try_index": t_idx
        })

    mcts_mean_points.append({
        "idx": i,
        "param_key": k,
        "series": "MCTS moyenne",
        "value": np.mean(try_vals) if len(try_vals) > 0 else np.nan
    })

df_mcts = pd.DataFrame(mcts_points)
df_mcts_mean = pd.DataFrame(mcts_mean_points)
df_solver = pd.DataFrame(solver_points)

fig = go.Figure()

if not df_mcts.empty:
    fig.add_trace(go.Scatter(
        x=df_mcts["idx"],
        y=df_mcts["value"],
        mode="markers",
        name="MCTS (tous les essais)",
        marker=dict(size=7, opacity=0.45),
        customdata=np.stack([df_mcts["param_key"], df_mcts["try_index"]], axis=-1),
        hovertemplate=(
            "<b>MCTS try</b><br>"
            "Paramètres: %{customdata[0]}<br>"
            "Try: %{customdata[1]}<br>"
            "Score normalisé: %{y:.4f}<extra></extra>"
        )
    ))

fig.add_trace(go.Scatter(
    x=df_mcts_mean["idx"],
    y=df_mcts_mean["value"],
    mode="lines+markers",
    name="MCTS (moyenne)",
    line=dict(width=2),
    marker=dict(size=8),
    customdata=np.stack([df_mcts_mean["param_key"]], axis=-1),
    hovertemplate=(
        "<b>MCTS moyenne</b><br>"
        "Paramètres: %{customdata[0]}<br>"
        "Score normalisé: %{y:.4f}<extra></extra>"
    )
))

fig.add_trace(go.Scatter(
    x=df_solver["idx"],
    y=df_solver["value"],
    mode="lines+markers",
    name="Solver",
    line=dict(width=2, dash="dash"),
    marker=dict(size=9, symbol="square"),
    customdata=np.stack([df_solver["param_key"]], axis=-1),
    hovertemplate=(
        "<b>Solver</b><br>"
        "Paramètres: %{customdata[0]}<br>"
        "Score normalisé: %{y:.4f}<extra></extra>"
    )
))

fig.update_layout(
    title="Scores MCTS vs Solver (interactif)",
    xaxis_title="Configuration (index)",
    yaxis_title="Score normalisé (MCTS / Solver)",
    template="plotly_dark",
    hovermode="closest",
    height=650
 )

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(len(param_keys))),
    ticktext=param_keys
 )

# Rendu navigateur: interactif avec hover sans dépendre du renderer notebook
fig.show(renderer="browser")